# QASMBench data discovery

This notebook investigates the supplied QASMBench Bronze archive.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import hashlib
import json
import re

import pandas as pd
from IPython.display import display

archive_path = Path(
    "/course-data/raw/source=qasmbench/qasmbench-qec.zip"
)
manifest_path = Path("/course-data/metadata/bundle-manifest.json")

assert archive_path.is_file(), "Archive not found: {}".format(archive_path)
assert manifest_path.is_file(), "Manifest not found: {}".format(manifest_path)

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
expected_object = next(
    item for item in manifest["objects"]
    if item["source"] == "qasmbench"
)

hasher = hashlib.sha256()
with archive_path.open("rb") as file:
    while True:
        chunk = file.read(1024 * 1024)
        if chunk == b"":
            break
        hasher.update(chunk)

actual_sha256 = hasher.hexdigest()
actual_bytes = archive_path.stat().st_size

integrity = pd.Series({
    "expected_bytes": expected_object["bytes"],
    "actual_bytes": actual_bytes,
    "size_matches": actual_bytes == expected_object["bytes"],
    "expected_sha256": expected_object["sha256"],
    "actual_sha256": actual_sha256,
    "checksum_matches": actual_sha256 == expected_object["sha256"],
})
display(integrity.to_frame("value"))

assert integrity["size_matches"]
assert integrity["checksum_matches"]


## Bronze integrity

- The archive contains 144,172 bytes, matching the manifest.
- The SHA-256 checksum also matches, confirming that Bronze is unchanged before parsing.


In [ ]:
def is_safe_member_path(name):
    normalized = name.replace("\\", "/")
    member_path = PurePosixPath(normalized)
    has_drive = bool(
        member_path.parts
        and re.fullmatch(r"[A-Za-z]:", member_path.parts[0])
    )
    return (
        bool(member_path.parts)
        and not member_path.is_absolute()
        and ".." not in member_path.parts
        and not has_drive
    )

with ZipFile(archive_path) as archive:
    member_names = archive.namelist()
    bad_crc_member = archive.testzip()
    member_inventory = pd.DataFrame([
        {
            "member": info.filename,
            "original_bytes": info.file_size,
            "compressed_bytes": info.compress_size,
            "safe_path": is_safe_member_path(info.filename),
        }
        for info in archive.infolist()
    ])

qasm_members = sorted(
    name for name in member_names
    if name.lower().endswith(".qasm")
)
unsafe_members = [
    name for name in member_names
    if not is_safe_member_path(name)
]

archive_summary = pd.Series({
    "all_members": len(member_names),
    "qasm_members": len(qasm_members),
    "unsafe_members": len(unsafe_members),
    "duplicate_member_names": len(member_names) - len(set(member_names)),
    "first_bad_crc_member": (
        bad_crc_member
        if bad_crc_member is not None
        else "No CRC errors"
    ),
})

display(member_inventory)
display(archive_summary.to_frame("value"))

assert len(qasm_members) == 6
assert not unsafe_members
assert len(member_names) == len(set(member_names))
assert bad_crc_member is None


## Archive contents and path safety

- The archive contains 19 members, including 6 QASM files.
- All paths are safe and unique.
- The ZIP CRC check found no damaged members.


In [ ]:
member_pattern = re.compile(
    r"^small/(?P<benchmark>[A-Za-z0-9_]+)/"
    r"(?P<filename>[A-Za-z0-9_]+)\.qasm$"
)

def qasm_member_metadata(member):
    match = member_pattern.fullmatch(member)
    if match is None:
        return {
            "member_valid": False,
            "benchmark_name": None,
            "variant": None,
        }

    benchmark_name = match.group("benchmark")
    filename = match.group("filename")
    if filename == benchmark_name:
        variant = "source"
    elif filename == benchmark_name + "_transpiled":
        variant = "transpiled"
    else:
        variant = None

    return {
        "member_valid": variant is not None,
        "benchmark_name": benchmark_name,
        "variant": variant,
    }

qasm_sources = {}
member_details = []
with ZipFile(archive_path) as archive:
    for member in qasm_members:
        member_bytes = archive.read(member)
        source_text = member_bytes.decode("utf-8")
        metadata = qasm_member_metadata(member)
        qasm_sources[member] = source_text
        member_details.append({
            "archive_member": member,
            "member_bytes": len(member_bytes),
            "member_sha256": hashlib.sha256(member_bytes).hexdigest(),
            "source_lines": len(source_text.splitlines()),
            **metadata,
        })

qasm_inventory = pd.DataFrame(member_details)
display(qasm_inventory)
display(
    qasm_inventory.groupby(["benchmark_name", "variant"])
    .size()
    .rename("files")
    .to_frame()
)

assert qasm_inventory["member_valid"].all()
assert set(qasm_inventory["variant"]) == {"source", "transpiled"}
assert qasm_inventory["benchmark_name"].nunique() == 3
assert qasm_inventory["member_sha256"].is_unique


## Filenames and variants

- The three benchmark families are `error_correctiond3_n5`, `qec_en_n5`, and `qec_sm_n5`.
- Each family contains one source circuit and one `_transpiled` circuit.
- Each file hash is retained for stable identification and source tracing.


In [ ]:
from quantum_lake_student.qasmbench import (
    extract_stabilizer_checks,
    parse_qasm,
)

parsed_by_member = {}
circuit_details = []

# Parsing
for member, source_text in qasm_sources.items():
    parsed = parse_qasm(source_text)
    checks = extract_stabilizer_checks(parsed)
    parsed_by_member[member] = parsed
    metadata = qasm_member_metadata(member)
    circuit_details.append({
        "archive_member": member,
        "benchmark_name": metadata["benchmark_name"],
        "variant": metadata["variant"],
        "register_declarations": parsed.register_declarations,
        "qubit_count": parsed.qubit_count,
        "classical_bit_count": parsed.classical_bit_count,
        "operation_count": parsed.operation_count,
        "measurement_count": parsed.measurement_count,
        "two_qubit_gate_count": parsed.two_qubit_gate_count,
        "custom_gate_definitions": len(parsed.gate_definitions),
        "stabilizer_checks": len(checks),
        "conditional_corrections": len(parsed.corrections),
    })

circuit_inventory = pd.DataFrame(circuit_details)
display(circuit_inventory)

assert (circuit_inventory["qubit_count"] == 5).all()
assert (circuit_inventory["measurement_count"] == 5).all()
assert (
    circuit_inventory["operation_count"]
    >= circuit_inventory["two_qubit_gate_count"]
).all()


## Registers and executed operations

- One circuit row represents one parsed source or transpiled QASM file.
- Register declarations preserve register names and sizes, so `q[0]` and `a[0]` remain different qubits.
- Operation counts include expanded unconditional gates; measurements and conditional corrections are counted separately.
- A custom gate body is counted only when that gate is called.


In [ ]:
qec_sm_source_member = (
    "small/qec_sm_n5/qec_sm_n5.qasm"
)
qec_sm_source = qasm_sources[qec_sm_source_member]
qec_sm_parsed = parsed_by_member[qec_sm_source_member]
qec_sm_checks = extract_stabilizer_checks(qec_sm_parsed)

print("Source QASM with line numbers:")
for line_number, line in enumerate(qec_sm_source.splitlines(), start=1):
    print("{:>2}: {}".format(line_number, line))

check_details = pd.DataFrame([
    {
        "ancilla_qubit": check.ancilla_qubit,
        "data_qubits": list(check.data_qubits),
        "syndrome_bit": check.syndrome_bit,
        "source_lines": list(check.source_lines),
    }
    for check in qec_sm_checks
])
correction_details = pd.DataFrame([
    {
        "condition_register": correction.condition_register,
        "condition_value": correction.condition_value,
        "gate": correction.gate,
        "target_qubit": correction.target_qubit,
        "source_line": correction.line_number,
    }
    for correction in qec_sm_parsed.corrections
])

display(check_details)
display(correction_details)

assert len(qec_sm_checks) == 2
assert len(qec_sm_parsed.corrections) == 3
assert {check.ancilla_qubit for check in qec_sm_checks} == {"a[0]", "a[1]"}
assert {check.syndrome_bit for check in qec_sm_checks} == {"syn[0]", "syn[1]"}
assert "q[0]" != "a[0]"


## Explicit parity checks and recovery

- `a[0]` checks `q[0]` and `q[1]`, producing `syn[0]`.
- `a[1]` checks `q[1]` and `q[2]`, producing `syn[1]`.
- Syndrome values 1, 2, and 3 apply X corrections to `q[0]`, `q[2]`, and `q[1]`.
- The `q` and `a` register names keep data and ancilla qubits distinct.


In [ ]:
comparison = circuit_inventory.pivot(
    index="benchmark_name",
    columns="variant",
    values=[
        "operation_count",
        "measurement_count",
        "two_qubit_gate_count",
    ],
)
display(comparison)

source_expected_counts = {
    "error_correctiond3_n5": {
        "operation_count": 114,
        "two_qubit_gate_count": 49,
    },
    "qec_en_n5": {
        "operation_count": 25,
        "two_qubit_gate_count": 10,
    },
    "qec_sm_n5": {
        "operation_count": 5,
        "two_qubit_gate_count": 4,
    },
}
for benchmark_name, expected_counts in source_expected_counts.items():
    row = circuit_inventory.loc[
        (circuit_inventory["benchmark_name"] == benchmark_name)
        & (circuit_inventory["variant"] == "source")
    ].iloc[0]
    assert row["operation_count"] == expected_counts["operation_count"]
    assert (
        row["two_qubit_gate_count"]
        == expected_counts["two_qubit_gate_count"]
    )

assert (
    comparison[("measurement_count", "source")]
    == comparison[("measurement_count", "transpiled")]
).all()
assert (
    comparison[("two_qubit_gate_count", "source")]
    == comparison[("two_qubit_gate_count", "transpiled")]
).all()


## Source versus transpiled circuits

- Measurement and two-qubit-gate counts remain unchanged after transpilation.
- Operations increase from 114 to 237 for `error_correctiond3_n5` and from 25 to 46 for `qec_en_n5`.
- `qec_sm_n5` remains at 5 operations.
- Source and transpiled files are separate variants because their code and file hashes differ.


In [ ]:
from quantum_lake_student.qasmbench import (
    BRONZE_OBJECT,
    SOURCE_NAME,
    build_qasmbench_tables,
)
from quantum_lake_student.source_validation import source_spec

qasmbench_spec = source_spec(
    manifest,
    SOURCE_NAME,
    expected_object=BRONZE_OBJECT,
)
tables = build_qasmbench_tables(
    archive_path.read_bytes(),
    spec=qasmbench_spec,
    run_id="qasmbench-discovery",
)

display(tables.circuits.to_pandas())
display(tables.stabilizer_checks.to_pandas())
display(tables.conditional_corrections.to_pandas())

all_silver_ids = (
    tables.circuits.column("source_record_id").to_pylist()
    + tables.stabilizer_checks.column("source_record_id").to_pylist()
    + tables.conditional_corrections.column("source_record_id").to_pylist()
)
trace_ids = tables.source_trace.column("source_record_id").to_pylist()

identifier_summary = pd.Series({
    "circuit_rows": tables.circuits.num_rows,
    "stabilizer_check_rows": tables.stabilizer_checks.num_rows,
    "conditional_correction_rows": tables.conditional_corrections.num_rows,
    "total_silver_rows": tables.silver_row_count,
    "trace_rows": tables.source_trace.num_rows,
    "issue_rows": tables.data_issues.num_rows,
    "source_record_ids_unique": len(all_silver_ids) == len(set(all_silver_ids)),
    "trace_covers_every_silver_row": set(all_silver_ids) == set(trace_ids),
})
display(identifier_summary.to_frame("value"))

assert tables.input_count == 6
assert tables.accepted_count == 6
assert tables.rejected_count == 0
assert tables.circuits.num_rows == 6
assert tables.stabilizer_checks.num_rows == 4
assert tables.conditional_corrections.num_rows == 6
assert tables.source_trace.num_rows == tables.silver_row_count
assert tables.data_issues.num_rows == 0
assert len(all_silver_ids) == len(set(all_silver_ids))
assert set(all_silver_ids) == set(trace_ids)


## Silver entities and trace coverage

- A circuit row represents one QASM variant, a stabilizer-check row represents one parity check, and a correction row represents one syndrome-controlled operation.
- Silver contains 6 circuit rows, 4 stabilizer-check rows, and 6 correction rows.
- All 16 identifiers are unique and every Silver row has a Bronze trace.
- No circuit was rejected and no data issue was recorded.


In [ ]:
qasm_fields = {
    "register_declarations",
    "operation_count",
    "measurement_count",
    "two_qubit_gate_count",
    "ancilla_qubit",
    "data_qubits",
    "syndrome_bit",
    "condition_register",
    "condition_value",
    "gate",
    "target_qubit",
}
experiment_fields = {
    "experiment_id",
    "physical_fault_rate",
    "shot_index",
    "logical_error_label",
}

relationship_evidence = pd.Series({
    "shared_experiment_fields": sorted(qasm_fields & experiment_fields),
    "contains_shot_records": False,
    "contains_physical_fault_rates": False,
    "contains_cross_source_experiment_id": False,
})
display(relationship_evidence.to_frame("value"))


## Rejected cross-source relationship

- QASMBench contains circuit structure but no shots, physical fault rates, or shared experiment identifiers.
- It cannot be joined row by row to the syndrome or Google datasets.
- The sources should share QEC terminology in Gold without inventing record-level links.


In [ ]:
discovery_summary = pd.Series({
    "bronze_checksum_valid": integrity["checksum_matches"],
    "safe_archive_paths": not unsafe_members,
    "qasm_files": len(qasm_members),
    "benchmarks": qasm_inventory["benchmark_name"].nunique(),
    "circuit_rows": tables.circuits.num_rows,
    "stabilizer_check_rows": tables.stabilizer_checks.num_rows,
    "conditional_correction_rows": tables.conditional_corrections.num_rows,
    "invalid_circuits": tables.rejected_count,
    "data_issues": tables.data_issues.num_rows,
    "trace_complete": set(all_silver_ids) == set(trace_ids),
})
display(discovery_summary.to_frame("value"))


## Final conclusions

- Bronze integrity, archive safety, and all six QASM members passed validation.
- The parser preserves register identity and counts expanded executed operations.
- The `qec_sm_n5` variants contain two stabilizer checks and three conditional corrections each.
- Silver contains 16 valid, uniquely identified, fully traced rows and no issues.
- No supported row-level relationship exists between QASMBench and the experiment datasets.
